In [4]:
pip install scikit-optimize


   ---------------------------------------- 0/2 [pyaml]
   -------------------- ------------------- 1/2 [scikit-optimize]
   -------------------- ------------------- 1/2 [scikit-optimize]
   -------------------- ------------------- 1/2 [scikit-optimize]
   ---------------------------------------- 2/2 [scikit-optimize]

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [3]:
"""
=============================================================================
Walk-Forward Validation & Hyperparameter Tuning — Kalman xG State-Space Model
met Asymmetrische Pi-Rating gewogen Kalman gain
=============================================================================

Hyperparameter vector θ = (σ²_w_att, σ²_w_def, σ²_v, φ, η, κ_att, κ_def, home_adv)

Pi-rating feature
------------------
Gebruikt pi_rating_diff_norm uit kalman_expected_goals.csv:
  pi_rating_diff_norm = (home_pi_home_pre - away_pi_away_pre - μ) / σ

Berekend via penaltyblog PiRatingSystem (Eastwood, 2025 / Constantinou & Fenton, 2013):
  alpha=0.15, beta=0.10, k=0.75 (package defaults)

Asymmetrische Kalman gain weging
----------------------------------
De informatiewaarde van een xG observatie voor aanval en verdediging
is asymmetrisch afhankelijk van tegenstander kwaliteit
(analoog aan opponent-strength-weighted updating in Glickman, 1999).
Lineup strength heeft geen rol in de gain-gewichten; dat effect loopt
uitsluitend via η in de observatievergelijking (xG-voorspelling zelf).

  pi_diff_norm > 0 → thuisploeg is sterker dan uitploeg

  w_att_h = clip(1 + κ_att · (−pi_diff_norm), 0.1, 3.0)
  w_def_a = clip(1 + κ_def · ( pi_diff_norm), 0.1, 3.0)
  w_att_a = clip(1 + κ_att · ( pi_diff_norm), 0.1, 3.0)
  w_def_h = clip(1 + κ_def · (−pi_diff_norm), 0.1, 3.0)

Optimalisatie — Optuna TPE
---------------------------
Optuna Tree-structured Parzen Estimator (TPE) bouwt na elke trial een
probabilistisch model van de zoekruimte en concentreert samples in
belovende regio's. Na Optuna volgt een L-BFGS-B polish stap om het
lokale minimum te verfijnen.

Walk-forward configuratie (Tashman, 2000)
------------------------------------------
Strikte scheiding tussen validatie (hyperparameter tuning) en test:

  VALIDATIE (expanding window, 6 folds):
    - min_train_seasons = 3  →  fold 1 traint op 3 seizoenen, valideert op seizoen 4
    - Trainen t/m 2023/24, valideren op elk volgend seizoen
    - Doel: stabiele hyperparameter schatting

  TEST (eenmalig, volledig gescheiden):
    - Getraind op alle data t/m 2023/24
    - Parameters = gemiddelde fold-specifieke schattingen
    - Geëvalueerd op 2024/25 en 2025/26
    - Testdata wordt nooit aangeraakt tijdens tuning

Referenties
-----------
Akiba, T. et al. (2019). Optuna: A next-generation hyperparameter optimization
  framework. Proceedings of the 25th ACM SIGKDD, 2623–2631.
Constantinou, A. & Fenton, N. (2013). Pi-ratings. JQAS.
Eastwood, M. (2025). Pi Ratings: The Smarter Way to Rank Football Teams.
  https://pena.lt/y/2025/04/14/pi-ratings-the-smarter-way-to-rank-football-teams/
Glickman, M.E. (1999). Parameter estimation in large dynamic paired comparison
  experiments. Applied Statistics, 48(3), 377–394.
Koopman, S.J. & Lit, R. (2015). Dynamic Bivariate Poisson Model for EPL.
Tashman, L.J. (2000). Out-of-sample tests of forecasting accuracy: an analysis
  and review. International Journal of Forecasting, 16(4), 437–450.
"""

import numpy as np
import pandas as pd
from scipy.stats import norm
import warnings
import json
import optuna
from datetime import datetime
from scipy.optimize import minimize

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

# ── Tuning configuratie ───────────────────────────────────────────────────────
N_TRIALS = 400

# Seizoenen die strikt als testset gelden — nooit aangeraakt tijdens tuning
TEST_SEASONS = {"2024/2025", "2025/2026"}


# ─────────────────────────────────────────────────────────────────────────────
# 1.  DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────

def load_data(
    path: str = r"C:/Users/semwi/FPL-Core-Insights_Thesis/data/kalman_expected_goals.csv",
) -> pd.DataFrame:
    """
    Laad data uit kalman_expected_goals.csv (penaltyblog pi-rating versie).

    Verwacht kolommen:
      pi_rating_diff_norm      : genormaliseerd verschil home_pi_home - away_pi_away
      lineup_strength_diff_norm: genormaliseerd lineup sterkte verschil
      home_xg, away_xg         : observed xG per wedstrijd
      home_team, away_team     : teamnamen
      season                   : seizoen string
    """
    df = pd.read_csv(path, parse_dates=["date"])
    df = df.sort_values("date").reset_index(drop=True)

    required = ["pi_rating_diff_norm", "lineup_strength_diff_norm",
                "home_xg", "away_xg", "home_team", "away_team", "season"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"Ontbrekende kolommen: {missing}\n"
            f"Draai eerst kalman_updater_v4.py om het bestand te genereren."
        )

    df["pi_home"] = df["pi_rating_diff_norm"]
    df["pi_away"] = df["pi_rating_diff_norm"]

    return df


# ─────────────────────────────────────────────────────────────────────────────
# 2.  KALMAN FILTER MET ASYMMETRISCHE PI-WEGING
# ─────────────────────────────────────────────────────────────────────────────

class KalmanXGFilter:
    """
    Bivariate Extended Kalman Filter (EKF) met log-link observatievergelijking
    en asymmetrische Pi-rating gewogen Kalman gain.

    State vector per team: (α, γ) = (aanvalskracht, verdedigingszwakte)
    Observatievergelijking:
      xG_home = exp(home_adv + α_home − γ_away + η·ls_norm) + v
      xG_away = exp(           α_away − γ_home − η·ls_norm) + v
    """

    def __init__(self, sigma2_w_att: float, sigma2_w_def: float,
                 sigma2_v: float, phi: float, eta: float,
                 kappa_att: float = 0.0, kappa_def: float = 0.0,
                 home_adv: float = 0.15):
        self.sigma2_w_att = max(sigma2_w_att, 1e-8)
        self.sigma2_w_def = max(sigma2_w_def, 1e-8)
        self.sigma2_v     = max(sigma2_v, 1e-8)
        self.phi          = np.clip(phi, 0.01, 0.999)
        self.eta          = eta
        self.kappa_att    = kappa_att
        self.kappa_def    = kappa_def
        self.home_adv     = home_adv
        self.states: dict = {}

    def _init_team(self, team: str):
        if team not in self.states:
            self.states[team] = {
                "alpha": 0.0, "gamma": 0.0,
                "var_alpha": 1.0, "var_gamma": 1.0,
            }

    def _predict_step(self, team: str):
        s = self.states[team]
        return (
            self.phi * s["alpha"],
            self.phi * s["gamma"],
            self.phi**2 * s["var_alpha"] + self.sigma2_w_att,
            self.phi**2 * s["var_gamma"] + self.sigma2_w_def,
        )

    def _weights(self, pi_diff_norm: float):
        w_att_h = 1.0 + self.kappa_att * (-pi_diff_norm)
        w_def_a = 1.0 + self.kappa_def * ( pi_diff_norm)
        w_att_a = 1.0 + self.kappa_att * ( pi_diff_norm)
        w_def_h = 1.0 + self.kappa_def * (-pi_diff_norm)
        return (
            float(np.clip(w_att_h, 0.1, 3.0)),
            float(np.clip(w_def_a, 0.1, 3.0)),
            float(np.clip(w_att_a, 0.1, 3.0)),
            float(np.clip(w_def_h, 0.1, 3.0)),
        )

    def predict_xg(self, home: str, away: str,
                   lineup_diff_norm: float,
                   pi_home: float = 0.0, pi_away: float = 0.0):
        self._init_team(home); self._init_team(away)
        ah, _, vah, _    = self._predict_step(home)
        aa, ga, vaa, vga = self._predict_step(away)
        _, gh, _, vgh    = self._predict_step(home)

        xg_home = np.exp(np.clip(self.home_adv + ah - ga + self.eta * lineup_diff_norm, -10, 10))
        xg_away = np.exp(np.clip(                aa - gh - self.eta * lineup_diff_norm, -10, 10))

        S_home = xg_home**2 * (vah + vga) + self.sigma2_v
        S_away = xg_away**2 * (vaa + vgh) + self.sigma2_v

        return xg_home, xg_away, S_home, S_away

    def update(self, home: str, away: str,
               obs_xg_home: float, obs_xg_away: float,
               lineup_diff_norm: float,
               pi_home: float = 0.0, pi_away: float = 0.0):
        self._init_team(home); self._init_team(away)

        ah_pred, gh_pred, vah_pred, vgh_pred = self._predict_step(home)
        aa_pred, ga_pred, vaa_pred, vga_pred = self._predict_step(away)

        xg_home_pred = np.exp(np.clip(self.home_adv + ah_pred - ga_pred + self.eta * lineup_diff_norm, -10, 10))
        xg_away_pred = np.exp(np.clip(                aa_pred - gh_pred - self.eta * lineup_diff_norm, -10, 10))

        e_home = obs_xg_home - xg_home_pred
        e_away = obs_xg_away - xg_away_pred

        G_home = xg_home_pred**2 * (vah_pred + vga_pred) + self.sigma2_v
        G_away = xg_away_pred**2 * (vaa_pred + vgh_pred) + self.sigma2_v

        ll = (
            -0.5 * np.log(2 * np.pi * G_home) - 0.5 * e_home**2 / G_home
            - 0.5 * np.log(2 * np.pi * G_away) - 0.5 * e_away**2 / G_away
        )

        K_ah = xg_home_pred * vah_pred / G_home
        K_ga = xg_home_pred * vga_pred / G_home
        K_aa = xg_away_pred * vaa_pred / G_away
        K_gh = xg_away_pred * vgh_pred / G_away

        w_att_h, w_def_a, w_att_a, w_def_h = self._weights(pi_home)

        self.states[home]["alpha"] = ah_pred + K_ah * e_home * w_att_h
        self.states[away]["gamma"] = ga_pred - K_ga * e_home * w_def_a
        self.states[away]["alpha"] = aa_pred + K_aa * e_away * w_att_a
        self.states[home]["gamma"] = gh_pred - K_gh * e_away * w_def_h

        self.states[home]["var_alpha"] = max((1 - K_ah * xg_home_pred) * vah_pred, 1e-8)
        self.states[home]["var_gamma"] = max((1 - K_gh * xg_away_pred) * vgh_pred, 1e-8)
        self.states[away]["var_alpha"] = max((1 - K_aa * xg_away_pred) * vaa_pred, 1e-8)
        self.states[away]["var_gamma"] = max((1 - K_ga * xg_home_pred) * vga_pred, 1e-8)

        return ll

    def run_on_sequence(self, df_seq: pd.DataFrame) -> float:
        total_ll = 0.0
        for _, row in df_seq.iterrows():
            ll = self.update(
                home=row["home_team"], away=row["away_team"],
                obs_xg_home=row["home_xg"], obs_xg_away=row["away_xg"],
                lineup_diff_norm=row["lineup_strength_diff_norm"],
                pi_home=row.get("pi_home", 0.0),
                pi_away=row.get("pi_away", 0.0),
            )
            if np.isfinite(ll):
                total_ll += ll
        return total_ll


# ─────────────────────────────────────────────────────────────────────────────
# 3.  METRICS
# ─────────────────────────────────────────────────────────────────────────────

def evaluate_predictions(df_eval: pd.DataFrame, kf: KalmanXGFilter) -> dict:
    ph, pa, sh, sa = [], [], [], []
    ah = df_eval["home_xg"].values
    aa = df_eval["away_xg"].values

    for _, row in df_eval.iterrows():
        xgh, xga, Gh, Ga = kf.predict_xg(
            row["home_team"], row["away_team"],
            row["lineup_strength_diff_norm"],
            pi_home=row.get("pi_home", 0.0),
            pi_away=row.get("pi_away", 0.0),
        )
        ph.append(xgh); pa.append(xga)
        sh.append(np.sqrt(Gh)); sa.append(np.sqrt(Ga))

    ph, pa = np.array(ph), np.array(pa)
    sh, sa = np.array(sh), np.array(sa)

    def mae(p, a):    return np.mean(np.abs(p - a))
    def rmse(p, a):   return np.sqrt(np.mean((p - a)**2))
    def nll(p, s, a): return -np.mean(norm.logpdf(a, loc=p, scale=np.maximum(s, 1e-6)))

    return {
        "mae_home":   mae(ph, ah),  "mae_away":   mae(pa, aa),
        "rmse_home":  rmse(ph, ah), "rmse_away":  rmse(pa, aa),
        "nll_home":   nll(ph, sh, ah), "nll_away": nll(pa, sa, aa),
        "mae_total":  mae(np.r_[ph, pa], np.r_[ah, aa]),
        "rmse_total": rmse(np.r_[ph, pa], np.r_[ah, aa]),
        "nll_total":  0.5 * (nll(ph, sh, ah) + nll(pa, sa, aa)),
        "n_obs": len(df_eval),
    }


# ─────────────────────────────────────────────────────────────────────────────
# 4.  OPTUNA OBJECTIVE
# ─────────────────────────────────────────────────────────────────────────────

def make_objective(df_train: pd.DataFrame):
    """
    Geeft een Optuna objective functie terug voor de gegeven trainingsset.
    Alle 8 parameters zijn vrij te optimaliseren.
    """
    def objective(trial: optuna.Trial) -> float:
        log_sw_att  = trial.suggest_float("log_sw_att",  -11.5,  0.69)
        log_sw_def  = trial.suggest_float("log_sw_def",  -11.5,  0.69)
        log_sv      = trial.suggest_float("log_sv",      -11.5,  1.61)
        phi_raw     = trial.suggest_float("phi_raw",       0.00,  6.90)
        eta         = trial.suggest_float("eta",          -1.00,  1.00)
        kap_att     = trial.suggest_float("kap_att",      -1.00,  2.00)
        kap_def     = trial.suggest_float("kap_def",      -1.00,  2.00)
        home_adv    = trial.suggest_float("home_adv",      0.00,  0.50)

        kf = KalmanXGFilter(
            sigma2_w_att  = np.exp(log_sw_att),
            sigma2_w_def  = np.exp(log_sw_def),
            sigma2_v      = np.exp(log_sv),
            phi           = 1.0 / (1.0 + np.exp(-phi_raw)),
            eta           = eta,
            kappa_att     = kap_att,
            kappa_def     = kap_def,
            home_adv      = home_adv,
        )
        ll = kf.run_on_sequence(df_train)
        return -ll if np.isfinite(ll) else 1e10

    return objective


def decode_optuna_params(params: dict) -> dict:
    """Vertaal Optuna parameter dict naar KalmanXGFilter kwargs."""
    return {
        "sigma2_w_att" : float(np.exp(params["log_sw_att"])),
        "sigma2_w_def" : float(np.exp(params["log_sw_def"])),
        "sigma2_v"     : float(np.exp(params["log_sv"])),
        "phi"          : float(1.0 / (1.0 + np.exp(-params["phi_raw"]))),
        "eta"          : float(params["eta"]),
        "kappa_att"    : float(params["kap_att"]),
        "kappa_def"    : float(params["kap_def"]),
        "home_adv"     : float(params["home_adv"]),
    }


# ─────────────────────────────────────────────────────────────────────────────
# 5.  WALK-FORWARD VALIDATOR (tuning only, geen testseizoenen)
# ─────────────────────────────────────────────────────────────────────────────

class WalkForwardValidator:
    """
    Expanding-window walk-forward validatie conform Tashman (2000).

    Testseizoenen (2024/25 en 2025/26) worden volledig uitgesloten
    van de validatiefolds — ze worden nooit aangeraakt tijdens tuning.
    """

    PARAM_COLS = ["sigma2_w_att", "sigma2_w_def", "sigma2_v",
                  "phi", "eta", "kappa_att", "kappa_def", "home_adv"]

    def __init__(self, df, min_train_seasons=3, n_folds=6, n_trials=N_TRIALS):
        self.df      = df.copy()

        # Sluit testseizoen strikt uit van validatie
        val_df = df[~df["season"].isin(TEST_SEASONS)]
        self.seasons = sorted(val_df["season"].unique())

        self.min_train_seasons = min_train_seasons
        self.n_folds           = n_folds
        self.n_trials          = n_trials
        self.fold_results      = []
        self.optimal_params_per_fold = []

    def _optimise(self, df_train: pd.DataFrame, fold_seed: int) -> dict:
        """Optuna TPE globale zoekstap + L-BFGS-B polish."""

        # Stap 1: Optuna TPE
        study = optuna.create_study(
            direction="minimize",
            sampler=optuna.samplers.TPESampler(seed=fold_seed),
        )
        study.optimize(
            make_objective(df_train),
            n_trials=self.n_trials,
            show_progress_bar=True,
        )
        best_optuna = decode_optuna_params(study.best_params)

        # Stap 2: L-BFGS-B polish
        bounds = [
            (-11.5, 0.69),  # log σ²_w_att
            (-11.5, 0.69),  # log σ²_w_def
            (-11.5, 1.61),  # log σ²_v
            ( 0.00, 6.90),  # φ_raw
            (-1.00, 1.00),  # η
            (-1.00, 2.00),  # κ_att
            (-1.00, 2.00),  # κ_def
            ( 0.00, 0.50),  # home_adv
        ]

        x0 = [
            np.log(best_optuna["sigma2_w_att"]),
            np.log(best_optuna["sigma2_w_def"]),
            np.log(best_optuna["sigma2_v"]),
            np.log(best_optuna["phi"] / (1.0 - best_optuna["phi"])),
            best_optuna["eta"],
            best_optuna["kappa_att"],
            best_optuna["kappa_def"],
            best_optuna["home_adv"],
        ]

        def neg_ll_polish(params):
            log_sa, log_sd, log_sv, phi_r, eta, katt, kdef, hadv = params
            kf = KalmanXGFilter(
                sigma2_w_att  = np.exp(log_sa),
                sigma2_w_def  = np.exp(log_sd),
                sigma2_v      = np.exp(log_sv),
                phi           = 1.0 / (1.0 + np.exp(-phi_r)),
                eta           = eta,
                kappa_att     = katt,
                kappa_def     = kdef,
                home_adv      = hadv,
            )
            ll = kf.run_on_sequence(df_train)
            return -ll if np.isfinite(ll) else 1e10

        res = minimize(neg_ll_polish, x0, method="L-BFGS-B", bounds=bounds,
                       options={"maxiter": 500, "ftol": 1e-10})

        log_sa, log_sd, log_sv, phi_r, eta, katt, kdef, hadv = res.x
        return {
            "sigma2_w_att" : float(np.exp(log_sa)),
            "sigma2_w_def" : float(np.exp(log_sd)),
            "sigma2_v"     : float(np.exp(log_sv)),
            "phi"          : float(1.0 / (1.0 + np.exp(-phi_r))),
            "eta"          : float(eta),
            "kappa_att"    : float(katt),
            "kappa_def"    : float(kdef),
            "home_adv"     : float(hadv),
        }

    def run(self, verbose=True) -> pd.DataFrame:
        fold_records = []

        available_folds = list(range(self.min_train_seasons, len(self.seasons)))
        if len(available_folds) > self.n_folds:
            fold_indices = available_folds[-self.n_folds:]
        else:
            fold_indices = available_folds

        print(f"\n   Walk-forward validatie (tuning only — testseizoenen uitgesloten)")
        print(f"   Testseizoen: {sorted(TEST_SEASONS)}")
        print(f"   Validatiefolds: {len(fold_indices)}")
        print(f"   Train start   : {self.seasons[0]}")
        print(f"   Val seizoenen : {[self.seasons[i] for i in fold_indices]}")
        print(f"   Optuna trials per fold: {self.n_trials}")

        for fold_num, val_idx in enumerate(fold_indices, 1):
            train_seasons = self.seasons[:val_idx]
            val_season    = self.seasons[val_idx]
            df_train = self.df[self.df["season"].isin(train_seasons)].copy()
            df_val   = self.df[self.df["season"] == val_season].copy()

            if verbose:
                print(f"\n{'='*65}")
                print(f"  Fold {fold_num}/{len(fold_indices)}")
                print(f"  Train: {train_seasons[0]} → {train_seasons[-1]}"
                      f"  ({len(df_train)} wedstrijden)")
                print(f"  Valid: {val_season}  ({len(df_val)} wedstrijden)")
                print(f"{'='*65}")
                print(f"  Optimising θ via Optuna TPE ({self.n_trials} trials) + L-BFGS-B polish...")

            t0      = datetime.now()
            theta   = self._optimise(df_train, fold_seed=42 + fold_num)
            elapsed = (datetime.now() - t0).total_seconds()

            if verbose:
                print(f"  Klaar in {elapsed:.1f}s")
                print(f"  θ* = σ²_w_att={theta['sigma2_w_att']:.5f}, "
                      f"σ²_w_def={theta['sigma2_w_def']:.5f}, "
                      f"σ²_v={theta['sigma2_v']:.5f}, φ={theta['phi']:.4f}")
                print(f"       η={theta['eta']:.4f}, "
                      f"κ_att={theta['kappa_att']:.4f}, "
                      f"κ_def={theta['kappa_def']:.4f}, "
                      f"home_adv={theta['home_adv']:.4f}")

            kf       = KalmanXGFilter(**theta)
            train_ll = kf.run_on_sequence(df_train)
            metrics  = evaluate_predictions(df_val, kf)

            fold_records.append({
                "fold"         : fold_num,
                "train_seasons": " | ".join(train_seasons),
                "val_season"   : val_season,
                "n_train"      : len(df_train),
                "n_val"        : len(df_val),
                "train_ll"     : train_ll,
                "opt_time_s"   : elapsed,
                **theta,
                **{f"val_{k}": v for k, v in metrics.items()},
            })
            self.optimal_params_per_fold.append(theta)

            if verbose:
                print(f"  Out-of-sample → MAE={metrics['mae_total']:.4f}, "
                      f"RMSE={metrics['rmse_total']:.4f}, "
                      f"NLL={metrics['nll_total']:.4f}")

        self.fold_results = pd.DataFrame(fold_records)
        return self.fold_results

    def mean_params(self) -> dict:
        """Gemiddelde hyperparameters over alle validatiefolds."""
        return {p: float(self.fold_results[p].mean())
                for p in self.PARAM_COLS}


# ─────────────────────────────────────────────────────────────────────────────
# 6.  FINALE TEST EVALUATIE (eenmalig, volledig gescheiden)
# ─────────────────────────────────────────────────────────────────────────────

def run_final_test(df: pd.DataFrame, theta: dict, out_path: str, verbose=True):
    """
    Finale test evaluatie op 2024/25 en 2025/26.

    - Getraind op alle data t/m 2023/24 met gemiddelde fold-parameters
    - Testdata is nooit aangeraakt tijdens walk-forward tuning
    - Conform Tashman (2000): strikte scheiding validatie vs. test
    """
    train_seasons = sorted([s for s in df["season"].unique()
                            if s not in TEST_SEASONS])
    test_seasons  = sorted(TEST_SEASONS)

    df_train = df[df["season"].isin(train_seasons)].copy()
    df_test  = df[df["season"].isin(test_seasons)].copy()

    if verbose:
        print(f"\n{'='*65}")
        print(f"  FINALE TEST EVALUATIE")
        print(f"  Train: {train_seasons[0]} → {train_seasons[-1]}"
              f"  ({len(df_train)} wedstrijden)")
        print(f"  Test : {test_seasons}  ({len(df_test)} wedstrijden)")
        print(f"  Parameters: gemiddelde over validatiefolds")
        print(f"{'='*65}")

    # Train op alle data t/m 2023/24 met finale parameters
    kf = KalmanXGFilter(**theta)
    kf.run_on_sequence(df_train)

    # Evalueer per testseizoen en gecombineerd
    results = {}
    for season in test_seasons:
        df_s = df_test[df_test["season"] == season].copy()
        metrics = evaluate_predictions(df_s, kf)
        results[season] = metrics
        if verbose:
            print(f"  {season}: MAE={metrics['mae_total']:.4f}, "
                  f"RMSE={metrics['rmse_total']:.4f}, "
                  f"NLL={metrics['nll_total']:.4f}  (n={metrics['n_obs']})")

    # Gecombineerd over beide testseizoenen
    metrics_combined = evaluate_predictions(df_test, kf)
    results["combined"] = metrics_combined
    if verbose:
        print(f"  Gecombineerd: MAE={metrics_combined['mae_total']:.4f}, "
              f"RMSE={metrics_combined['rmse_total']:.4f}, "
              f"NLL={metrics_combined['nll_total']:.4f}  (n={metrics_combined['n_obs']})")

    # Sla testresultaten op
    with open(f"{out_path}final_test_results.json", "w") as f:
        json.dump(results, f, indent=2)

    if verbose:
        print(f"\n  Opgeslagen: {out_path}final_test_results.json")

    return results, kf


# ─────────────────────────────────────────────────────────────────────────────
# 7.  MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main():
    print("\n" + "="*65)
    print("  KALMAN xG — ASYMMETRISCHE PI-RATING GEWOGEN KALMAN GAIN")
    print("  Pi-rating: penaltyblog PiRatingSystem (Eastwood, 2025)")
    print("  Feature: pi_rating_diff_norm = (home_pi - away_pi - μ) / σ")
    print(f"  Optimisatie: Optuna TPE ({N_TRIALS} trials) + L-BFGS-B polish")
    print("  Walk-forward: 3 jaar min train, 6 folds (t/m 2023/24)")
    print(f"  Test: {sorted(TEST_SEASONS)} (volledig gescheiden)")
    print("  θ = (σ²_w_att, σ²_w_def, σ²_v, φ, η, κ_att, κ_def, home_adv)")
    print("="*65)

    DATA_PATH = r"C:/Users/semwi/FPL-Core-Insights_Thesis/data/kalman_expected_goals.csv"
    OUT_PATH  = r"C:/Users/semwi/FPL-Core-Insights_Thesis/data/Kalman data/"

    # Data laden
    df = load_data(DATA_PATH)
    print(f"\n   Loaded {len(df)} wedstrijden, {df['season'].nunique()} seizoenen")
    print(f"   Seizoenen: {sorted(df['season'].unique())}")
    print(f"   Testseizoen (uitgesloten van tuning): {sorted(TEST_SEASONS)}")

    # ── Stap 1: Walk-forward validatie voor hyperparameter tuning ──────────
    wfv     = WalkForwardValidator(df, min_train_seasons=3, n_folds=6, n_trials=N_TRIALS)
    results = wfv.run(verbose=True)

    pc = WalkForwardValidator.PARAM_COLS
    mc = ["val_mae_total", "val_rmse_total", "val_nll_total"]

    print("\n\n" + "="*65)
    print("  RESULTATEN PER VALIDATIEFOLD")
    print("="*65)
    print(results[["fold", "val_season", "n_train", "n_val"] + pc + mc].to_string(
          index=False, float_format=lambda x: f"{x:.5f}"))

    print("\n\n" + "="*65)
    print("  AGGREGATE OVER ALLE VALIDATIEFOLDS")
    print("="*65)
    print(results[mc].agg(["mean", "std", "min", "max"]).to_string(
          float_format=lambda x: f"{x:.5f}"))

    print("\n\n" + "="*65)
    print("  HYPERPARAMETER STABILITEIT")
    print("="*65)
    print(results[pc].agg(["mean", "std", "min", "max"]).to_string(
          float_format=lambda x: f"{x:.6f}"))

    # ── Stap 2: Finale test op 2024/25 + 2025/26 ──────────────────────────
    mean_theta = wfv.mean_params()
    print(f"\n  Finale parameters (gemiddelde over folds):")
    for k, v in mean_theta.items():
        print(f"    {k}: {v:.6f}")

    test_results, final_kf = run_final_test(df, mean_theta, OUT_PATH)

    # ── Opslaan ────────────────────────────────────────────────────────────
    results.to_csv(f"{OUT_PATH}wfv_fold_results.csv", index=False)

    with open(f"{OUT_PATH}optimal_hyperparameters.json", "w") as f:
        json.dump({
            "pi_mode"     : "penaltyblog_diff_norm",
            "pi_params"   : {"alpha": 0.15, "beta": 0.10, "k": 0.75},
            "optimiser"   : "optuna_tpe_plus_lbfgsb_polish",
            "n_trials"    : N_TRIALS,
            "n_folds"     : 6,
            "test_seasons": sorted(TEST_SEASONS),
            "mean"        : {p: float(results[p].mean()) for p in pc},
            "final_fold"  : {p: float(results.iloc[-1][p]) for p in pc},
            "per_fold"    : [{p: float(results.iloc[i][p]) for p in pc}
                             for i in range(len(results))],
        }, f, indent=2)

    print(f"\nOpgeslagen:")
    print(f"  {OUT_PATH}wfv_fold_results.csv")
    print(f"  {OUT_PATH}optimal_hyperparameters.json")
    print(f"  {OUT_PATH}final_test_results.json")

    return results, test_results


if __name__ == "__main__":
    results, test_results = main()


  KALMAN xG — ASYMMETRISCHE PI-RATING GEWOGEN KALMAN GAIN
  Pi-rating: penaltyblog PiRatingSystem (Eastwood, 2025)
  Feature: pi_rating_diff_norm = (home_pi - away_pi - μ) / σ
  Optimisatie: Optuna TPE (400 trials) + L-BFGS-B polish
  Walk-forward: 3 jaar min train, 6 folds (t/m 2023/24)
  Test: ['2024/2025', '2025/2026'] (volledig gescheiden)
  θ = (σ²_w_att, σ²_w_def, σ²_v, φ, η, κ_att, κ_def, home_adv)

   Loaded 3779 wedstrijden, 10 seizoenen
   Seizoenen: ['2016/2017', '2017/2018', '2018/2019', '2019/2020', '2020/2021', '2021/2022', '2022/2023', '2023/2024', '2024/2025', '2025/2026']
   Testseizoen (uitgesloten van tuning): ['2024/2025', '2025/2026']

   Walk-forward validatie (tuning only — testseizoenen uitgesloten)
   Testseizoen: ['2024/2025', '2025/2026']
   Validatiefolds: 5
   Train start   : 2016/2017
   Val seizoenen : ['2019/2020', '2020/2021', '2021/2022', '2022/2023', '2023/2024']
   Optuna trials per fold: 400

  Fold 1/5
  Train: 2016/2017 → 2018/2019  (1140 wedstri

  0%|          | 0/400 [00:00<?, ?it/s]

  Klaar in 190.9s
  θ* = σ²_w_att=0.00211, σ²_w_def=0.00159, σ²_v=0.42783, φ=0.9861
       η=-0.0245, κ_att=0.0390, κ_def=-0.3288, home_adv=0.2675
  Out-of-sample → MAE=0.6390, RMSE=0.8297, NLL=1.2986

  Fold 2/5
  Train: 2016/2017 → 2019/2020  (1520 wedstrijden)
  Valid: 2020/2021  (380 wedstrijden)
  Optimising θ via Optuna TPE (400 trials) + L-BFGS-B polish...


  0%|          | 0/400 [00:00<?, ?it/s]

  Klaar in 156.1s
  θ* = σ²_w_att=0.00134, σ²_w_def=0.00066, σ²_v=0.47085, φ=0.9913
       η=-0.0017, κ_att=0.1163, κ_def=-0.2431, home_adv=0.2292
  Out-of-sample → MAE=0.6317, RMSE=0.8127, NLL=1.2279

  Fold 3/5
  Train: 2016/2017 → 2020/2021  (1900 wedstrijden)
  Valid: 2021/2022  (380 wedstrijden)
  Optimising θ via Optuna TPE (400 trials) + L-BFGS-B polish...


...

  Klaar in 176.1s
  θ* = σ²_w_att=0.00125, σ²_w_def=0.00100, σ²_v=0.48176, φ=0.9900
       η=0.0202, κ_att=0.1094, κ_def=-0.2738, home_adv=0.2048
  Out-of-sample → MAE=0.6265, RMSE=0.7982, NLL=1.2189

  Fold 4/5
  Train: 2016/2017 → 2021/2022  (2280 wedstrijden)
  Valid: 2022/2023  (380 wedstrijden)
  Optimising θ via Optuna TPE (400 trials) + L-BFGS-B polish...


  0%|          | 0/400 [00:00<?, ?it/s]

  Klaar in 254.4s
  θ* = σ²_w_att=0.00113, σ²_w_def=0.00114, σ²_v=0.48285, φ=0.9915
       η=0.0387, κ_att=0.1019, κ_def=-0.2495, home_adv=0.2016
  Out-of-sample → MAE=0.6278, RMSE=0.8244, NLL=1.2634

  Fold 5/5
  Train: 2016/2017 → 2022/2023  (2660 wedstrijden)
  Valid: 2023/2024  (380 wedstrijden)
  Optimising θ via Optuna TPE (400 trials) + L-BFGS-B polish...


  0%|          | 0/400 [00:00<?, ?it/s]

  Klaar in 240.7s
  θ* = σ²_w_att=0.00124, σ²_w_def=0.00094, σ²_v=0.49183, φ=0.9910
       η=0.0477, κ_att=0.1145, κ_def=-0.2693, home_adv=0.2123
  Out-of-sample → MAE=0.7159, RMSE=0.9340, NLL=1.3781


  RESULTATEN PER VALIDATIEFOLD
 fold val_season  n_train  n_val  sigma2_w_att  sigma2_w_def  sigma2_v     phi      eta  kappa_att  kappa_def  home_adv  val_mae_total  val_rmse_total  val_nll_total
    1  2019/2020     1140    380       0.00211       0.00159   0.42783 0.98612 -0.02450    0.03905   -0.32883   0.26752        0.63901         0.82969        1.29860
    2  2020/2021     1520    380       0.00134       0.00066   0.47085 0.99133 -0.00171    0.11633   -0.24309   0.22921        0.63173         0.81267        1.22787
    3  2021/2022     1900    380       0.00125       0.00100   0.48176 0.99004  0.02022    0.10944   -0.27383   0.20477        0.62652         0.79821        1.21894
    4  2022/2023     2280    380       0.00113       0.00114   0.48285 0.99149  0.03869    0.10193   -0

In [1]:
"""
Dixon-Coles Parameter Tuning — Gevectoriseerde versie
======================================================
Tunet: SCALE_HOME, SCALE_AWAY, MAX_GOALS
RHO vastgezet op -0.13 conform Dixon & Coles (1997)
Objective: RPS minimalisatie via walk-forward validatie

Walk-forward configuratie (Tashman, 2000):
  - Testseizoen (2024/25 + 2025/26) volledig uitgesloten van tuning
  - Validatiefolds lopen t/m 2023/24
  - MIN_TRAIN = 3 seizoenen

RHO = -0.13: empirisch geschat door Dixon & Coles (1997) op Premier
League data. Geeft aan dat lage gelijke scores vaker voorkomen dan
het onafhankelijke Poisson model voorspelt. Breed gerepliceerd in
de literatuur en daarom vastgehouden als literatuurwaarde.
"""

import os
import numpy as np
import pandas as pd
from scipy.stats import poisson
from scipy.optimize import minimize, differential_evolution
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = r"C:\Users\semwi\FPL-Core-Insights_Thesis\data"
CSV_PATH = os.path.join(BASE_DIR, "kalman_expected_goals.csv")

TEST_SEASONS = {"2024/2025", "2025/2026"}

# RHO vastgezet op literatuurwaarde (Dixon & Coles, 1997)
RHO_FIXED = -0.13

# ── Data laden ────────────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH, parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)
df = df[df["kalman_xg_pred_home"].notna() & df["home_goals"].notna()].copy()
df["result"] = df.apply(
    lambda r: 'H' if r['home_goals'] > r['away_goals']
    else 'D' if r['home_goals'] == r['away_goals'] else 'A', axis=1)

df_tune = df[~df["season"].isin(TEST_SEASONS)].copy()

print(f"Totaal geladen:          {len(df)} wedstrijden")
print(f"Beschikbaar voor tuning: {len(df_tune)} wedstrijden")
print(f"Testseizoen uitgesloten: {sorted(TEST_SEASONS)}")
print(f"Seizoenen voor tuning:   {sorted(df_tune['season'].unique())}")
print(f"\nRHO vastgezet op {RHO_FIXED} (Dixon & Coles, 1997)")
print(f"\nEmpirische schalen (tuningset):")
print(f"  Scale home: {df_tune['home_xg'].mean() / df_tune['kalman_xg_pred_home'].mean():.4f}")
print(f"  Scale away: {df_tune['away_xg'].mean() / df_tune['kalman_xg_pred_away'].mean():.4f}")

# ══════════════════════════════════════════════════════════════════════════════
# GEVECTORISEERDE POISSON + DIXON-COLES
# ══════════════════════════════════════════════════════════════════════════════
def compute_probs(xg_h, xg_a, scale_h, scale_a, max_goals, rho=RHO_FIXED):
    lh  = np.maximum(xg_h * scale_h, 0.05)
    la  = np.maximum(xg_a * scale_a, 0.05)
    ph  = np.zeros(len(lh))
    pd_ = np.zeros(len(lh))
    pa  = np.zeros(len(lh))

    for i in range(max_goals + 1):
        for j in range(max_goals + 1):
            p = poisson.pmf(i, lh) * poisson.pmf(j, la)
            if   i == 0 and j == 0: p *= (1 - lh * la * rho)
            elif i == 1 and j == 0: p *= (1 + la * rho)
            elif i == 0 and j == 1: p *= (1 + lh * rho)
            elif i == 1 and j == 1: p *= (1 - rho)
            if   i > j:  ph  += p
            elif i == j: pd_ += p
            else:        pa  += p

    total = np.where(ph + pd_ + pa == 0, 1, ph + pd_ + pa)
    return ph/total, pd_/total, pa/total

def compute_rps(ph, pd_, pa, results):
    o1 = (results == 'H').astype(float)
    o2 = ((results == 'H') | (results == 'D')).astype(float)
    return 0.5 * ((ph - o1)**2 + (ph + pd_ - o2)**2)

def objective(params, df_eval):
    scale_h, scale_a, mg_raw = params
    max_goals = int(np.clip(round(mg_raw), 4, 12))
    ph, pd_, pa = compute_probs(
        df_eval["kalman_xg_pred_home"].values,
        df_eval["kalman_xg_pred_away"].values,
        scale_h, scale_a, max_goals
    )
    return compute_rps(ph, pd_, pa, df_eval["result"].values).mean()

# ══════════════════════════════════════════════════════════════════════════════
# WALK-FORWARD TUNING (alleen t/m 2023/24)
# ══════════════════════════════════════════════════════════════════════════════
seasons   = sorted(df_tune['season'].unique())
MIN_TRAIN = 3

bounds = [
    (0.99,1.00),  # scale_home
    (0.99,1.00),  # scale_away
    (4, 12),       # max_goals
]

print(f"\n{'='*65}")
print(f"  WALK-FORWARD TUNING — Dixon-Coles parameters")
print(f"  RHO = {RHO_FIXED} (Dixon & Coles, 1997) — niet getuund")
print(f"  Vrije parameters: SCALE_HOME, SCALE_AWAY, MAX_GOALS")
print(f"  Testseizoen uitgesloten: {sorted(TEST_SEASONS)}")
print(f"  Validatiefolds: t/m {seasons[-1]}")
print(f"{'='*65}")

fold_results = []

for val_idx in range(MIN_TRAIN, len(seasons)):
    train_seasons = seasons[:val_idx]
    val_season    = seasons[val_idx]
    df_train = df_tune[df_tune['season'].isin(train_seasons)].copy()
    df_val   = df_tune[df_tune['season'] == val_season].copy()

    print(f"\n  Fold {val_idx - MIN_TRAIN + 1}/{len(seasons) - MIN_TRAIN}")
    print(f"  Train: {train_seasons[0]} -> {train_seasons[-1]} ({len(df_train)} matches)")
    print(f"  Valid: {val_season} ({len(df_val)} matches)")

    # Stap 1: Differential Evolution globale zoekstap
    de = differential_evolution(
        objective, bounds, args=(df_train,),
        strategy='best1bin', maxiter=300, popsize=12,
        tol=1e-7, seed=42, polish=False, disp=False
    )

    # Stap 2: L-BFGS-B polish
    res = minimize(
        objective, de.x, args=(df_train,),
        method='L-BFGS-B', bounds=bounds,
        options={'maxiter': 1000, 'ftol': 1e-12}
    )

    scale_h, scale_a, mg_raw = res.x
    max_goals = int(np.clip(round(mg_raw), 4, 12))

    ph, pd_, pa = compute_probs(
        df_val["kalman_xg_pred_home"].values,
        df_val["kalman_xg_pred_away"].values,
        scale_h, scale_a, max_goals
    )
    rps_all = compute_rps(ph, pd_, pa, df_val["result"].values)
    val_rps = rps_all.mean()
    res_arr = df_val["result"].values
    rps_h   = rps_all[res_arr == 'H'].mean()
    rps_d   = rps_all[res_arr == 'D'].mean()
    rps_a   = rps_all[res_arr == 'A'].mean()

    print(f"  θ* = scale_h={scale_h:.4f}, scale_a={scale_a:.4f}, "
          f"max_goals={max_goals}  (rho={RHO_FIXED} fixed)")
    print(f"  Val RPS: overall={val_rps:.4f} | "
          f"H={rps_h:.4f} | D={rps_d:.4f} | A={rps_a:.4f}")

    fold_results.append({
        'fold':       val_idx - MIN_TRAIN + 1,
        'val_season': val_season,
        'scale_home': round(scale_h, 4),
        'scale_away': round(scale_a, 4),
        'rho':        RHO_FIXED,
        'max_goals':  max_goals,
        'val_rps':    round(val_rps, 5),
        'rps_h':      round(rps_h, 5),
        'rps_d':      round(rps_d, 5),
        'rps_a':      round(rps_a, 5),
    })

# ══════════════════════════════════════════════════════════════════════════════
# RESULTATEN
# ══════════════════════════════════════════════════════════════════════════════
results_df = pd.DataFrame(fold_results)

print(f"\n\n{'='*65}")
print(f"  RESULTATEN PER FOLD")
print(f"{'='*65}")
print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

means = results_df[['scale_home', 'scale_away', 'max_goals', 'val_rps']].mean()

print(f"\n\n{'='*65}")
print(f"  GEMIDDELDE OVER ALLE VALIDATIEFOLDS")
print(f"{'='*65}")
print(f"\n  scale_home = {means['scale_home']:.4f}")
print(f"  scale_away = {means['scale_away']:.4f}")
print(f"  rho        = {RHO_FIXED}  (Dixon & Coles, 1997)")
print(f"  max_goals  = {round(means['max_goals'])}")
print(f"  RPS        = {means['val_rps']:.5f}")

print(f"\n  -- Gebruik deze waarden in je scripts: --")
print(f"  SCALE_HOME = {means['scale_home']:.4f}")
print(f"  SCALE_AWAY = {means['scale_away']:.4f}")
print(f"  RHO        = {RHO_FIXED}  # Dixon & Coles (1997)")
print(f"  MAX_GOALS  = {round(means['max_goals'])}")

results_df.to_csv(os.path.join(BASE_DIR, "dixon_coles_tuning.csv"), index=False)
print(f"\n  Opgeslagen: {os.path.join(BASE_DIR, 'dixon_coles_tuning.csv')}")

Totaal geladen:          3779 wedstrijden
Beschikbaar voor tuning: 3040 wedstrijden
Testseizoen uitgesloten: ['2024/2025', '2025/2026']
Seizoenen voor tuning:   ['2016/2017', '2017/2018', '2018/2019', '2019/2020', '2020/2021', '2021/2022', '2022/2023', '2023/2024']

RHO vastgezet op -0.13 (Dixon & Coles, 1997)

Empirische schalen (tuningset):
  Scale home: 1.0246
  Scale away: 1.0278

  WALK-FORWARD TUNING — Dixon-Coles parameters
  RHO = -0.13 (Dixon & Coles, 1997) — niet getuund
  Vrije parameters: SCALE_HOME, SCALE_AWAY, MAX_GOALS
  Testseizoen uitgesloten: ['2024/2025', '2025/2026']
  Validatiefolds: t/m 2023/2024

  Fold 1/5
  Train: 2016/2017 -> 2018/2019 (1140 matches)
  Valid: 2019/2020 (380 matches)
  θ* = scale_h=1.0000, scale_a=0.9900, max_goals=12  (rho=-0.13 fixed)
  Val RPS: overall=0.2005 | H=0.1779 | D=0.1506 | A=0.2736

  Fold 2/5
  Train: 2016/2017 -> 2019/2020 (1520 matches)
  Valid: 2020/2021 (380 matches)
  θ* = scale_h=1.0000, scale_a=0.9900, max_goals=12  (rho=-0